<center> <img src = https://raw.githubusercontent.com/AndreyRysistov/DatasetsForPandas/main/hh%20label.jpg alt="drawing" style="width:400px;">

# <center> Проект: Анализ вакансий из HeadHunter
   

In [1]:
import pandas as pd
import psycopg2
import plotly.express as px

In [2]:
DBNAME = 'project_sql'
USER = 'skillfactory'
PASSWORD = 'cCkxxLVrDE8EbvjueeMedPKt'
HOST = '84.201.134.129'
PORT = 5432

In [3]:
conn = psycopg2.connect(
    dbname=DBNAME,
    user=USER,
    host=HOST,
    password=PASSWORD,
    port=PORT
    )

In [4]:
import warnings
warnings.filterwarnings('ignore')

# Юнит 3. Предварительный анализ данных

1. Напишите запрос, который посчитает количество вакансий в нашей базе (вакансии находятся в таблице vacancies). 

In [5]:
# текст запроса
query_3_1 = f'''
    SELECT COUNT(*) cnt FROM vacancies;
    '''

In [6]:
pd.read_sql_query(query_3_1, conn)

,cnt
0,49197


2. Напишите запрос, который посчитает количество работодателей (таблица employers). 

In [7]:
query_3_2 = f'''
    SELECT COUNT(*) cnt FROM employers;
    '''

In [8]:
pd.read_sql(query_3_2, conn)

,cnt
0,23501


3. Посчитате с помощью запроса количество регионов (таблица areas).

In [9]:
query_3_3 = f'''
    SELECT COUNT(*) cnt FROM areas;
    '''

In [10]:
pd.read_sql(query_3_3, conn)

,cnt
0,1362


4. Посчитате с помощью запроса количество сфер деятельности в базе (таблица industries).

In [11]:
query_3_4 = f'''
    SELECT COUNT(*) cnt FROM industries;
    '''

In [12]:
pd.read_sql(query_3_4, conn)

,cnt
0,294


***

Данные представлены в виде реляционной БД состоящей из основной таблицы вакансий (vacancies) и 4х справочных связанных с ней по ключам таблиц. Данная структура позволяет строить аналитику предлагаемых вакансий в разрезе работодателей, географических областей и сфер деятельности работодателей.

# Юнит 4. Детальный анализ вакансий

1. Напишите запрос, который позволит узнать, сколько (cnt) вакансий в каждом регионе (area).
Отсортируйте по количеству вакансий в порядке убывания.

In [13]:
query_4_1 = f'''
    SELECT
        a1.name area
        , COUNT(*) cnt
    FROM areas a1
        JOIN vacancies a2 ON a1.id = a2.area_id
    GROUP BY a1.name
    ORDER BY cnt DESC;
    '''

In [14]:
df_4_1=pd.read_sql(query_4_1, conn)
display(df_4_1)

,area,cnt
0,Москва,5333
1,Санкт-Петербург,2851
2,Минск,2112
3,Новосибирск,2006
4,Алматы,1892
...,...,...
764,Тарко-Сале,1
765,Новоаннинский,1
766,Бирск,1
767,Сасово,1


2. Напишите запрос, чтобы определить у какого количества вакансий заполнено хотя бы одно из двух полей с зарплатой.

In [16]:
query_4_2 = f'''
    SELECT COUNT(*) cnt
    FROM vacancies a1
    WHERE a1.salary_from IS NOT NULL OR a1.salary_to IS NOT NULL;
    '''

In [17]:
pd.read_sql(query_4_2, conn)

,cnt
0,24073


3. Найдите средние значения для нижней и верхней границы зарплатной вилки. Округлите значения до целого.

In [18]:
query_4_3 = f'''
    SELECT
        AVG(a1.salary_from) avg_from
        , AVG(a1.salary_to) avg_to
    FROM vacancies a1;
    '''

In [19]:
pd.read_sql(query_4_3, conn)

,avg_from,avg_to
0,71064.657901,110536.741923


4. Напишите запрос, который выведет количество вакансий для каждого сочетания типа рабочего графика (schedule) и типа трудоустройства (employment), используемого в вакансиях. Результат отсортируйте по убыванию количества.


In [20]:
query_4_4 = f'''
    SELECT
        a1.schedule
        , a1.employment
        , COUNT(*) cnt
    FROM vacancies a1
    GROUP BY a1.schedule, a1.employment
    ORDER BY cnt DESC;
    '''

In [21]:
pd.read_sql(query_4_4, conn)

,schedule,employment,cnt
0,Полный день,Полная занятость,35367
1,Удаленная работа,Полная занятость,7802
2,Гибкий график,Полная занятость,1593
3,Удаленная работа,Частичная занятость,1312
4,Сменный график,Полная занятость,940
5,Полный день,Стажировка,569
6,Вахтовый метод,Полная занятость,367
7,Полный день,Частичная занятость,347
8,Гибкий график,Частичная занятость,312
9,Полный день,Проектная работа,141


5. Напишите запрос, выводящий значения поля Требуемый опыт работы (experience) в порядке возрастания количества вакансий, в которых указан данный вариант опыта. 

In [22]:
query_4_5 = f'''
    SELECT
        a1.experience
        , COUNT(*) cnt
    FROM vacancies a1
    WHERE a1.experience IN (
        'Более 6 лет'
        , 'Нет опыта'
        , 'От 3 до 6 лет'
        , 'От 1 года до 3 лет')
    GROUP BY a1.experience
    ORDER BY cnt ASC;
    '''

In [23]:
pd.read_sql(query_4_5, conn)

,experience,cnt
0,Более 6 лет,1337
1,Нет опыта,7197
2,От 3 до 6 лет,14511
3,От 1 года до 3 лет,26152


***

База с широким охватом регионов, пропорциональным размеру городов количеством вакансий.  Разбивка по опыту близка к средним показателям карьерного роста в IT индустрии (junior,middle,senior).Основной спрос на сотрудников с начальным опытом, на полный рабочий день. Дельта между средними минимальной и максимальной ЗП составляет 50%. Основная концентрация вакансий в крупнейших городах, максимальная в Москве.  Из 769 регионов с вакансиями в 699 зарегистрировано менее 100 вакансий. В целом базу можно считать релевантной, отвечающей положению на рынке труда. Из отрицательных моментов можно отметить низкую заполненность вакансий информацией о заработной плате (~50%).

# Юнит 5. Анализ работодателей

1. Напишите запрос, который позволит узнать, какие работодатели находятся на первом и пятом месте по количеству вакансий.

In [24]:
query_5_1 = f'''
SELECT
    a1.name     employer
    , a1.cnt    vac_cnt
    , a1.rnm    posit
FROM (
    SELECT
        a1.name
        , COUNT(*) cnt
        , ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) rnm
    FROM employers a1
        LEFT JOIN vacancies a2 ON a1.id = a2.employer_id
    GROUP BY a1.name
    ) a1
WHERE a1.rnm IN (1,2,5);
'''

In [25]:
pd.read_sql(query_5_1, conn)

,employer,vac_cnt,posit
0,Яндекс,1933,1
1,Ростелеком,491,2
2,Газпром нефть,331,5


2. Напишите запрос, который для каждого региона выведет количество работодателей и вакансий в нём.
Среди регионов, в которых нет вакансий, найдите тот, в котором наибольшее количество работодателей.


In [26]:
query_5_2 = f'''
    SELECT
        a1.name         area
        , COUNT(a3.id)  empl_cnt
        , COUNT(a2.id)  vac_cnt
    FROM areas a1
        LEFT JOIN employers a3 ON a1.id = a3.area
        LEFT JOIN vacancies a2 ON a1.id = a2.area_id AND a3.id = a2.employer_id
    GROUP BY a1.name
    HAVING COUNT(a2.id) = 0
    ORDER BY empl_cnt DESC
    LIMIT 1;
    '''

In [27]:
pd.read_sql(query_5_2, conn)

,area,empl_cnt,vac_cnt
0,Россия,410,0


3. Для каждого работодателя посчитайте количество регионов, в которых он публикует свои вакансии. Отсортируйте результат по убыванию количества.


In [28]:
query_5_3 = f'''
    SELECT
        a1.name employer
        , COUNT(DISTINCT a2.area_id) area_cnt
    FROM employers a1
        LEFT JOIN vacancies a2 ON a1.id = a2.employer_id
    GROUP BY a1.name
    ORDER BY area_cnt DESC;
    '''

In [29]:
pd.read_sql(query_5_3, conn)

,employer,area_cnt
0,Яндекс,181
1,Ростелеком,152
2,Спецремонт,116
3,Поляков Денис Иванович,88
4,ООО ЕФИН,71
...,...,...
23170,СДЕЛКА,0
23171,Alandr Group,0
23172,СДК,0
23173,Lemon Land Lombard,0


4. Напишите запрос для подсчёта количества работодателей, у которых не указана сфера деятельности. 

In [30]:
query_5_4 = f'''
    SELECT
        COUNT(*) as cnt
    FROM employers a1
        LEFT JOIN EMPLOYERS_INDUSTRIES a2 ON a1.id = a2.employer_id
    WHERE a2.employer_id IS NULL;
    '''

In [31]:
pd.read_sql(query_5_4, conn)

,cnt
0,8419


5. Напишите запрос, чтобы узнать название компании, находящейся на третьем месте в алфавитном списке (по названию) компаний, у которых указано четыре сферы деятельности. 

In [32]:
query_5_5 = f'''
    WITH pre_slct AS (
        SELECT
            a1.name
            , COUNT(*) as cnt
            , ROW_NUMBER() OVER(ORDER BY a1.name ASC) rnm
        FROM employers a1
            LEFT JOIN EMPLOYERS_INDUSTRIES a2 ON a1.id = a2.employer_id
        GROUP BY a1.name
        HAVING COUNT(*) = 4
        )
    SELECT
        name employer
    FROM pre_slct
    WHERE rnm = 3;
    '''

In [33]:
pd.read_sql(query_5_5, conn)

,employer
0,2ГИС


6. С помощью запроса выясните, у какого количества работодателей в качестве сферы деятельности указана Разработка программного обеспечения.


In [34]:
query_5_6 = f'''
    SELECT
        COUNT(*) empl_cnt
    FROM employers a1
        LEFT JOIN employers_industries a2 ON a1.id = a2.employer_id
        LEFT JOIN industries a3 ON a2.industry_id = a3.id
    WHERE a3.name = 'Разработка программного обеспечения';
    '''

In [35]:
pd.read_sql(query_5_6, conn)

,empl_cnt
0,3553


7. Для компании «Яндекс» выведите список регионов-миллионников, в которых представлены вакансии компании, вместе с количеством вакансий в этих регионах. Также добавьте строку Total с общим количеством вакансий компании. Результат отсортируйте по возрастанию количества.

Список городов-милионников надо взять [отсюда](https://ru.wikipedia.org/wiki/%D0%93%D0%BE%D1%80%D0%BE%D0%B4%D0%B0-%D0%BC%D0%B8%D0%BB%D0%BB%D0%B8%D0%BE%D0%BD%D0%B5%D1%80%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8). 

Если возникнут трудности с этим задание посмотрите материалы модуля  PYTHON-17. Как получать данные из веб-источников и API. 

In [36]:
url = "https://ru.wikipedia.org/wiki/%D0%93%D0%BE%D1%80%D0%BE%D0%B4%D0%B0-%D0%BC%D0%B8%D0%BB%D0%BB%D0%B8%D0%BE%D0%BD%D0%B5%D1%80%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8"

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
    }

df = pd.read_html(url, storage_options=headers)
param_str = "'" + "','".join( df[0]['Город'].values ) + "'"

In [37]:
query_5_7 = f'''
    WITH pre_slct AS (
        SELECT
            a3.name
            , COUNT(*) as cnt
        FROM employers a1
            LEFT JOIN vacancies a2 ON a1.id = a2.employer_id
            LEFT JOIN areas a3 ON a2.area_id = a3.id
        WHERE a1.name = 'Яндекс'
            AND a3.name IN ({param_str})
        GROUP BY a3.name
        )
    SELECT
        name  area
        , cnt vac_cnt
    FROM pre_slct
    UNION ALL
    SELECT
        'Total'
        , SUM(cnt)
    FROM pre_slct
    ORDER BY vac_cnt ASC;
    '''

In [38]:
pd.read_sql(query_5_7, conn)

,area,vac_cnt
0,Омск,21.0
1,Челябинск,22.0
2,Красноярск,23.0
3,Волгоград,24.0
4,Пермь,25.0
5,Казань,25.0
6,Ростов-на-Дону,25.0
7,Самара,26.0
8,Уфа,26.0
9,Краснодар,30.0


***

Лидером по охвату регионов и количеству вакансий является Яндекс (1933 вакансий), в 4 раза опережая ближайшего конкурента Ростелеком (491 вакансия). В качестве сферы деятельности у 15% работодателей указана "разработка программного обеспечения. У 36% работодателей не указана сфера деятельности, что снижает качество данных. Можно отметить равномерное распределение вакансий компании Яндекс по городам миллионникам, что может характеризовать компанию как гибкую в распределения бизнеса по регионам.

# Юнит 6. Предметный анализ

1. Сколько вакансий имеет отношение к данным?

Считаем, что вакансия имеет отношение к данным, если в её названии содержатся слова 'data' или 'данн'.

*Подсказка: Обратите внимание, что названия вакансий могут быть написаны в любом регистре.* 


In [39]:
query_6_1 = f'''
    SELECT COUNT(*) cnt
    FROM vacancies a1
    WHERE LOWER(a1.name) LIKE '%data%'
        OR LOWER(a1.name) LIKE '%данн%';
    '''

In [40]:
pd.read_sql(query_6_1, conn)

,cnt
0,1771


2. Сколько есть подходящих вакансий для начинающего дата-сайентиста? 
Будем считать вакансиями для дата-сайентистов такие, в названии которых есть хотя бы одно из следующих сочетаний:
* 'data scientist'
* 'data science'
* 'исследователь данных'
* 'ML' (здесь не нужно брать вакансии по HTML)
* 'machine learning'
* 'машинн%обучен%'

** В следующих заданиях мы продолжим работать с вакансиями по этому условию.*

Считаем вакансиями для специалистов уровня Junior следующие:
* в названии есть слово 'junior' *или*
* требуемый опыт — Нет опыта *или*
* тип трудоустройства — Стажировка.
 

In [41]:
query_6_2 = f"""
    SELECT COUNT(*) cnt
    FROM vacancies a1
    WHERE (LOWER(a1.name) LIKE '%data scientist%'
        OR LOWER(a1.name) LIKE '%data science%'
        OR LOWER(a1.name) LIKE '%исследователь данных%'
        OR (LOWER(a1.name) LIKE '%ml%'
            AND LOWER(a1.name) NOT LIKE '%html%')
        OR LOWER(a1.name) LIKE '%machine learning%'
        OR LOWER(a1.name) LIKE '%машинн%обучен%')
    AND (LOWER(a1.name) LIKE '%junior%'
        OR a1.experience = 'Нет опыта'
        OR a1.employment = 'Стажировка');
    """

In [42]:
pd.read_sql(query_6_2, conn)

,cnt
0,51


3. Сколько есть вакансий для DS, в которых в качестве ключевого навыка указан SQL или postgres?

** Критерии для отнесения вакансии к DS указаны в предыдущем задании.*

In [45]:
query_6_3 = f"""
    SELECT COUNT(*) cnt
    FROM vacancies a1
    WHERE (LOWER(a1.name) LIKE '%data scientist%'
        OR LOWER(a1.name) LIKE '%data science%'
        OR LOWER(a1.name) LIKE '%исследователь данных%'
        OR (LOWER(a1.name) LIKE '%ml%'
            AND LOWER(a1.name) NOT LIKE '%html%')
        OR LOWER(a1.name) LIKE '%machine learning%'
        OR LOWER(a1.name) LIKE '%машинн%обучен%')
    AND (LOWER(a1.key_skills) LIKE '%sql%'
        OR LOWER(a1.key_skills) LIKE '%postgres%');
    """

In [46]:
pd.read_sql(query_6_3, conn)

,cnt
0,229


4. Проверьте, насколько популярен Python в требованиях работодателей к DS.Для этого вычислите количество вакансий, в которых в качестве ключевого навыка указан Python.

** Это можно сделать помощью запроса, аналогичного предыдущему.*

In [47]:
query_6_4 = f"""
    SELECT COUNT(*) cnt
    FROM vacancies a1
    WHERE (LOWER(a1.name) LIKE '%data scientist%'
        OR LOWER(a1.name) LIKE '%data science%'
        OR LOWER(a1.name) LIKE '%исследователь данных%'
        OR (LOWER(a1.name) LIKE '%ml%' AND LOWER(a1.name) NOT LIKE '%html%')
        OR LOWER(a1.name) LIKE '%machine learning%'
        OR LOWER(a1.name) LIKE '%машинн%обучен%')
    AND LOWER(a1.key_skills) LIKE '%python%';
    """

In [48]:
pd.read_sql(query_6_4, conn)

,cnt
0,357


5. Сколько ключевых навыков в среднем указывают в вакансиях для DS?
Ответ округлите до двух знаков после точки-разделителя.

In [49]:
query_6_5 = """
    SELECT
        AVG(LENGTH(a1.key_skills) - LENGTH(REPLACE(a1.key_skills, CHR(9), '')) + 1) avg_cnt
    FROM vacancies a1
    WHERE (LOWER(a1.name)  LIKE '%data scientist%'
        OR LOWER(a1.name)  LIKE '%data science%'
        OR LOWER(a1.name)  LIKE '%исследователь данных%'
        OR (a1.name LIKE '%ML%' AND a1.name NOT LIKE '%HTML%')
        OR LOWER(a1.name)  LIKE '%machine learning%'
        OR LOWER(a1.name)  LIKE '%машинн%обучен%');
    """

In [50]:
pd.read_sql(query_6_5, conn)

,avg_cnt
0,6.406032


6. Напишите запрос, позволяющий вычислить, какую зарплату для DS в **среднем** указывают для каждого типа требуемого опыта (уникальное значение из поля *experience*). 

При решении задачи примите во внимание следующее:
1. Рассматриваем только вакансии, у которых заполнено хотя бы одно из двух полей с зарплатой.
2. Если заполнены оба поля с зарплатой, то считаем зарплату по каждой вакансии как сумму двух полей, делённую на 2. Если заполнено только одно из полей, то его и считаем зарплатой по вакансии.
3. Если в расчётах участвует null, в результате он тоже даст null (посмотрите, что возвращает запрос select 1 + null). Чтобы избежать этой ситуацию, мы воспользуемся функцией [coalesce](https://postgrespro.ru/docs/postgresql/9.5/functions-conditional#functions-coalesce-nvl-ifnull), которая заменит null на значение, которое мы передадим. Например, посмотрите, что возвращает запрос `select 1 + coalesce(null, 0)`

Выясните, на какую зарплату в среднем может рассчитывать дата-сайентист с опытом работы от 3 до 6 лет. Результат округлите до целого числа.

In [51]:
query_6_6 = """
    SELECT
        a1.experience
        , AVG((COALESCE(a1.salary_from, a1.salary_to)+COALESCE(a1.salary_to, a1.salary_from))/2) salary_avg
    FROM vacancies a1
    WHERE (LOWER(a1.name) LIKE '%data scientist%'
        OR LOWER(a1.name) LIKE '%data science%'
        OR LOWER(a1.name) LIKE '%исследователь данных%'
        OR (a1.name LIKE '%ML%'
            AND a1.name NOT LIKE '%HTML%')
        OR LOWER(a1.name) LIKE '%machine learning%'
        OR LOWER(a1.name) LIKE '%машинн%обучен%')
    GROUP BY a1.experience;
    """

In [52]:
pd.read_sql(query_6_6, conn)

,experience,salary_avg
0,Более 6 лет,NaN
1,Нет опыта,74642.857143
2,От 1 года до 3 лет,139674.750000
3,От 3 до 6 лет,243114.666667


***

Из представленных вакансий в ~ 3,6% связаны с обработкой данных. DS вакансии составляют 1,1% (536),</br>только 10% (51) из них уровня Junior.
    В среднем в качестве требований для вакансий в сфере DS указано 6 навыков, из них:</br>
    - сочетание знания SQL и Postgres, 42%;</br>
    - знание Python, 66,6%.

Из проведенного анализа можно сделать вывод, что предложение вакансий связанных с обработкой данных и в частности DS невелико, однако уровень ЗП для соискателей с опытом значительно выше среднего. Одно из основных требований для вакансий в сфере DS это знание Python и SQL Postrges.

# Общий вывод по проекту

В целом БД можно признать пригодной для задачи поиска вакансий в сфере IT, к недостаткам можно отнести отсутствие информации по предлагаемой ЗП у значительной части вакансий. Следует систематизировать требования в части навыков, выделить в отдельную таблицу, для более точного анализа и поиска.
Из представленных данных следует что наиболее крупный работодатель на рынке IT является Яндекс, максимальное количество предлагаемых вакансий в крупнейших городах России. На момент формирования базы рынок DS занимал незначительную часть, однако специалисты оценивались выше среднего с хорошей перспективой роста ЗП. Знание языка программирования Python является одним из основных требований к соискателям на должность DS, что подтверждает ценность данного курса.

In [43]:
#  дополнительно, общее количество вакансий DS
query_6_2_1 = f"""
    SELECT COUNT(*) cnt
    FROM vacancies a1
    WHERE (LOWER(a1.name) LIKE '%data scientist%'
        OR LOWER(a1.name) LIKE '%data science%'
        OR LOWER(a1.name) LIKE '%исследователь данных%'
        OR (LOWER(a1.name) LIKE '%ml%'
            AND LOWER(a1.name) NOT LIKE '%html%')
        OR LOWER(a1.name) LIKE '%machine learning%'
        OR LOWER(a1.name) LIKE '%машинн%обучен%');
    """

In [44]:
pd.read_sql(query_6_2_1, conn)

,cnt
0,536
